# 💻 DataMate — Zalo AI Challenge 2026: Local GPU Test & Inference

Notebook này dành riêng để **chạy trực tiếp trên máy tính cá nhân (Local)** có GPU NVIDIA (RTX 30xx / 40xx / 50xx).
- Không cần kết nối Colab hay Google Drive.
- Tự động sử dụng dữ liệu thư mục `public_test` ngay tại máy.

### 1. Kiểm Tra Thông Tin GPU & Môi Trường PyTorch

In [ ]:
# 1.1 Kiểm tra driver NVIDIA qua lệnh hệ thống
!nvidia-smi

In [ ]:
# 1.2 Kiểm tra PyTorch đã nhận GPU CUDA chưa
import torch
print("PyTorch Version:", torch.__version__)
cuda_avail = torch.cuda.is_available()
print("CUDA Available :", cuda_avail)
if cuda_avail:
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print(f"VRAM Dung lượng : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("⚠️ CẢNH BÁO: PyTorch chưa nhận CUDA GPU. Hãy chạy ô 2.1 bên dưới để cài bản PyTorch có CUDA.")

### 2. Cài Đặt Thư Viện Cần Thiết
*(Nếu `CUDA Available` ở trên là `False`, hãy chạy ô 2.1 trước để cài PyTorch hỗ trợ GPU CUDA)*

In [ ]:
# 2.1 (Tùy chọn) Cài đặt PyTorch hỗ trợ CUDA 12.4 / 12.6 nếu chưa có
# Bỏ comment dòng dưới để chạy nếu cần:
# !pip install --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cu124

In [ ]:
# 2.2 Cài đặt các thư viện xử lý dữ liệu và AI cần thiết
!pip install -q duckdb>=1.1.0 pandas>=2.2.0 openpyxl python-calamine
!pip install -q transformers>=4.48.0 accelerate>=1.0.0 bitsandbytes>=0.43.0
!pip install -q tqdm loguru unidecode regex

### 3. Kiểm Tra File Dữ Liệu & Mã Nguồn Local

In [ ]:
import os

print("Thư mục hiện tại       :", os.path.abspath(os.getcwd()))
print("public_test/ có sẵn   :", os.path.exists("public_test"))
print("src/ có sẵn           :", os.path.exists("src"))
print("run_public_test.py    :", os.path.exists("run_public_test.py"))
print("calendar.csv có sẵn   :", os.path.exists(os.path.join("public_test", "calendar.csv")))
print("questions.jsonl có sẵn:", os.path.exists(os.path.join("public_test", "questions.jsonl")))

### 4. Chạy Thử Nghiệm 5 Câu (Sanity Check)
Chạy thử 5 câu đầu tiên với mô hình `Qwen/Qwen2.5-Coder-7B-Instruct` ở chế độ nén 4-bit (chỉ tốn ~5.5GB VRAM).

In [ ]:
# Xóa logs cũ trước khi chạy
import shutil
shutil.rmtree("result/logs", ignore_errors=True)

# Chạy 5 câu đầu tiên
!python run_public_test.py \
    --data_dir public_test \
    --output submission_sample.jsonl \
    --model_id Qwen/Qwen2.5-Coder-7B-Instruct \
    --load_in_4bit \
    --limit 5

In [ ]:
# Xem kết quả của 5 câu mẫu
!type submission_sample.jsonl

### 5. Xem Chi Tiết Log Suy Luận & Code Đã Chạy Của Từng Câu

In [ ]:
import glob, json, os

log_files = sorted(glob.glob("result/logs/*.json"))
print(f"Tổng số câu đã chạy: {len(log_files)}\n")
for f in log_files:
    d = json.load(open(f, encoding="utf-8"))
    sub = d.get("final_submission", {})
    print(f"🔹 [{d['id']}] Q: {d['question']}")
    print(f"   Đáp án : {sub.get('type')} -> {sub.get('answer')}")
    print(f"   Thời gian: {d.get('elapsed_time')}s")
    if d.get("exec_error"):
        print(f"   ⚠️ Lỗi thực thi: {d.get('exec_error')}")
    print("-" * 60)

### 6. Chạy Toàn Bộ 1,000 Câu Hỏi (Full Public Test)
Chạy khi bạn đã sẵn sàng nộp bài. Kết quả được lưu liên tục vào file `submission.jsonl`.

In [ ]:
!python run_public_test.py \
    --data_dir public_test \
    --output submission.jsonl \
    --model_id Qwen/Qwen2.5-Coder-7B-Instruct \
    --load_in_4bit

### 7. Thống Kê File `submission.jsonl` Trước Khi Nộp

In [ ]:
import os, json
from collections import Counter

if os.path.exists("submission.jsonl"):
    lines = []
    with open("submission.jsonl", "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                lines.append(json.loads(line))
    print(f"✅ Tổng số dòng: {len(lines)} / 1000")
    types = Counter([x.get("type") for x in lines])
    print("Phân bố các loại đáp án:")
    for t, c in types.most_common():
        pct = (c / len(lines)) * 100
        print(f"  {t:15s}: {c:4d} ({pct:5.1f}%)")
else:
    print("Chưa có file submission.jsonl")